# 🧠 Notebook 14: LLM-Powered HyDE & Clinical Query Expansion
## Multilingual Medical Document Retrieval (ViBioMIR 2026)

Notebook này triển khai kỹ thuật **HyDE (Hypothetical Document Embeddings)** và **Clinical Entity Expansion** sử dụng mô hình ngôn ngữ lớn (LLM) để thu hẹp khoảng cách ngữ nghĩa giữa **câu hỏi triệu chứng của bệnh nhân** và **bài viết y khoa chuyên sâu của bác sĩ**.

### 🎯 Mục tiêu chính:
1. **Tải tập 1,200 câu hỏi kiểm thử (`query.parquet`)**.
2. **Cấu hình LLM tối ưu trên Kaggle GPU (T4 16GB)**: Dùng `Qwen/Qwen2.5-3B-Instruct` hoặc `Qwen/Qwen2.5-7B-Instruct` (4-bit quantization với `bitsandbytes` để suy luận siêu tốc không tràn VRAM).
3. **Thực thi 2 chiến lược sinh văn bản y khoa**:
   - **HyDE Document**: Sinh đoạn văn bản y khoa giả định mô tả bệnh lý, triệu chứng, xét nghiệm và hướng xử trí.
   - **Clinical Keywords**: Trích xuất tên các bệnh nghi ngờ (Chẩn đoán phân biệt), thuật ngữ y khoa Anh - Việt và từ khóa chuyên ngành.
4. **Cơ chế lưu Checkpoint tự động (`run.partial.jsonl`)**: Chống mất mát dữ liệu nếu kết nối bị ngắt giữa chừng.
5. **Xuất file chuẩn bị sẵn (`data/processed/queries_hyde.jsonl`)**: Giúp quá trình inference chính thức chạy offline trong **0 giây**!

### 1. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pyarrow pandas tqdm huggingface_hub

### 2. Kiểm tra GPU và Cấu hình Môi trường

In [ ]:
import os
import json
import time
from pathlib import Path
import torch
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Kiểm tra GPU khả dụng
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị tính toán: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM)")

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HYDE_OUTPUT_PATH = OUTPUT_DIR / "queries_hyde.jsonl"
CHECKPOINT_PATH = OUTPUT_DIR / "queries_hyde.partial.jsonl"

### 3. Nạp tập câu hỏi kiểm thử (`query.parquet`)

In [ ]:
import glob
from huggingface_hub import hf_hub_download

queries_candidates = glob.glob("/kaggle/input/**/query*.parquet", recursive=True) + glob.glob("data/raw/**/query*.parquet", recursive=True)

if queries_candidates:
    query_path = queries_candidates[0]
    print(f"✓ Tìm thấy queries local: {query_path}")
else:
    print("🌐 Đang tải query.parquet từ Hugging Face: AIGuruTinix/ViBioMIR...")
    query_path = hf_hub_download(
        repo_id="AIGuruTinix/ViBioMIR",
        filename="query.parquet",
        repo_type="dataset",
        local_dir="data/raw/vibio_mir",
    )

query_df = pq.read_table(query_path).to_pandas()
print(f"✅ Đã nạp thành công {len(query_df)} câu hỏi!")
print("Mẫu câu hỏi #1:", query_df.iloc[0].to_dict())

### 4. Khởi tạo Mô hình LLM Y Khoa với 4-bit Quantization

Ta sử dụng mô hình **`Qwen/Qwen2.5-3B-Instruct`** (hoặc `Qwen/Qwen2.5-7B-Instruct`). Dòng mô hình Qwen 2.5 có khả năng hiểu tiếng Việt và kiến thức y học cực kỳ xuất sắc, chạy rất mượt và nhẹ trên Kaggle GPU.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # Có thể đổi sang "Qwen/Qwen2.5-7B-Instruct" nếu có GPU 16GB

print(f"⏳ Đang nạp Tokenizer & Model {MODEL_NAME} (4-bit NF4)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Cấu hình 4-bit quantization qua BitsAndBytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True,
)
model.eval()
print("✓ Nạp mô hình LLM thành công! Sẵn sàng sinh HyDE.")

### 5. Thiết kế Prompt Chuyên Khoa Y Tế

Prompt được tối ưu hóa để vừa sinh **Hypothetical Document (Tài liệu giả định)**, vừa trích xuất **Clinical Entities & Diagnoses** nhằm tránh hiện tượng ảo giác (hallucination).

In [ ]:
SYSTEM_PROMPT = """Bạn là một chuyên gia y khoa, bác sĩ cố vấn cao cấp. Nhiệm vụ của bạn là hỗ trợ hệ thống tra cứu tài liệu y khoa.
Khi nhận được câu hỏi hoặc mô tả triệu chứng của người bệnh, bạn hãy thực hiện 2 nhiệm vụ ngắn gọn:
1. CHẨN ĐOÁN & THUẬT NGỮ: Liệt kê các bệnh lý nghi ngờ (kèm tên tiếng Anh / mã ICD nếu biết) và thuật ngữ chuyên ngành liên quan.
2. TÀI LIỆU GIẢ ĐỊNH (HyDE): Viết một đoạn văn y khoa ngắn (~100-150 từ) mô tả cơ chế bệnh sinh, triệu chứng lâm sàng đặc trưng, phương pháp chẩn đoán cận lâm sàng và nguyên tắc điều trị cho trường hợp này.

Hãy trả lời bằng tiếng Việt chuyên môn, súc tích, không thêm lời chào hỏi rườm rà."""

def build_user_prompt(query_text: str) -> str:
    return f"""Câu hỏi của người bệnh:
\"{query_text}\"

Hãy phân tích và viết tóm tắt y khoa giả định theo cấu trúc:
- Bệnh lý nghi ngờ & Thuật ngữ: ...
- Tóm tắt y khoa: ..."""

# Tự động phát hiện tên cột chứa câu hỏi (thường là 'query' hoặc 'text')
query_col = 'query' if 'query' in query_df.columns else ('text' if 'text' in query_df.columns else query_df.columns[1])
print(f"Cột câu hỏi được sử dụng: '{query_col}'")

# Kiểm thử nhanh với 1 câu hỏi mẫu
sample_q = query_df.iloc[0][query_col]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": build_user_prompt(sample_q)}
]
prompt_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_input, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )
sample_res = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== KẾT QUẢ THỬ NGHIỆM MẪU ===")
print(sample_res)

### 6. Chạy Batch Suy Luận cho Toàn Bộ 1,200 Câu Hỏi (Có Checkpoint)

In [ ]:
# 1. Đọc lại checkpoint nếu đã chạy trước đó
completed_records = {}
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                completed_records[item["id"]] = item
    print(f"✓ Đã nạp {len(completed_records)} câu hỏi từ checkpoint cũ!")

# 2. Chạy suy luận cho các câu hỏi còn lại
remaining_queries = [row for _, row in query_df.iterrows() if row["id"] not in completed_records]
print(f"Số lượng câu hỏi cần xử lý: {len(remaining_queries)} / {len(query_df)}")

checkpoint_file = open(CHECKPOINT_PATH, "a", encoding="utf-8")
t_start = time.time()

try:
    for row in tqdm(remaining_queries, desc="🚀 Generating HyDE & Clinical Expansions"):
        qid = int(row["id"])
        qtext = str(row[query_col]).strip()
        
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(qtext)}
        ]
        prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer(prompt_str, return_tensors="pt").to(device)
        
        with torch.no_grad():
            gen_ids = model.generate(
                **model_inputs,
                max_new_tokens=250,
                temperature=0.2,
                top_p=0.9,
                do_sample=True,
                repetition_penalty=1.1,
            )
        generated_text = tokenizer.decode(gen_ids[0][model_inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Ghép văn bản mở rộng (Original + HyDE Generated)
        enriched_text = f"{qtext}\n\n[Thông tin chuyên khoa]:\n{generated_text}"
        
        record = {
            "id": qid,
            "original_text": qtext,
            "hyde_expansion": generated_text,
            "enriched_query": enriched_text,
        }
        
        # Ghi trực tiếp vào file checkpoint
        checkpoint_file.write(json.dumps(record, ensure_ascii=False) + "\n")
        checkpoint_file.flush()
        completed_records[qid] = record

finally:
    checkpoint_file.close()

total_elapsed = time.time() - t_start
print(f"\n🎉 HOÀN TẤT SINH HYDE TRONG {total_elapsed / 60:.1f} PHÚT!")

### 7. Xuất file Hoàn chỉnh (`queries_hyde.jsonl`)

In [ ]:
# Sắp xếp lại theo thứ tự ID tuần tự 1..1200
all_records = [completed_records[qid] for qid in sorted(completed_records.keys())]

with open(HYDE_OUTPUT_PATH, "w", encoding="utf-8") as f:
    for rec in all_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"✅ ĐÃ XUẤT THÀNH CÔNG: {HYDE_OUTPUT_PATH}")
print(f"• Dung lượng file: {HYDE_OUTPUT_PATH.stat().st_size / (1024*1024):.2f} MB")
print(f"• Số lượng câu hỏi: {len(all_records)}")

### 8. Hướng dẫn Tích hợp vào Pipeline Suy Luận

Sau khi có file `data/processed/queries_hyde.jsonl`, trong notebook chạy inference chính thức (`inference-queries.ipynb`), bạn chỉ cần truyền cờ `--queries` trỏ đến file này:

```bash
python scripts/run_inference_only.py \
    --queries data/processed/queries_hyde.jsonl \
    --hybrid-top-k 650 \
    --top-k-docs 300 \
    --top-k-chunks 150 \
    --max-chunks-per-doc 3 \
    --expand-top-docs 45
```

👉 **Kết quả**: BGE-M3 Dense Search và BM25s sẽ đọc các văn bản đã được bổ sung kiến thức y khoa chuyên sâu từ LLM, giúp tìm kiếm chính xác tài liệu bài viết gấp nhiều lần mà thời gian chạy suy luận trên Kaggle không tốn thêm bất kỳ giây nào!